# Zebrafish fin, H3K27ac ChIP-seq, 0 vs 1 day post-amputation (Wang W et al. 2020; PRJNA559885)
Result 1 (AP-1 spacing, replication set). Kaggle notebook: **Internet ON, CPU** (about 3-5 h).

Steps: FASTQ from ENA (MD5-checked) -> fastp -> Bowtie 2 -> MAPQ >= 30, duplicates removed -> MACS3 broad peaks against input -> RREs and control pool with control signal >= median (`01_call_injury_responsive_enhancers.py`).

Note: in the original run the enhancer call with the median control threshold was performed locally on this notebook's BAM and peak files with the same script; it is placed here so the pipeline runs end to end. The spacing tests for this set are run locally with step 03 (see README).

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null && apt-get -qq install -y bowtie2 samtools bedtools unzip > /dev/null
pip -q install macs3
wget -q http://opengene.org/fastp/fastp -O /usr/local/bin/fastp && chmod +x /usr/local/bin/fastp
for t in twoBitToFa faToTwoBit; do
  wget -q https://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/$t -O /usr/local/bin/$t && chmod +x /usr/local/bin/$t
done
bowtie2 --version | head -1; macs3 --version; nproc; free -g | head -2

In [ ]:
%%writefile 01_call_injury_responsive_enhancers.py
"""Result 1, step 01: injury-responsive enhancers (RREs) and the control pool from H3K27ac ChIP-seq.

Input : MACS3 broadPeak files per sample, filtered BAM per sample (uninjured R1 R2, injured R1 R2), TSS BED.
Consensus peaks = merged peaks of all samples; signal = CPM per kb per replicate.
RRE    : log2FC(injured/uninjured) >= 1 in BOTH replicate pairs (R1-R1, R2-R2), injured signal >= 75th percentile,
         > 2 kb from the nearest TSS.
Control: |mean log2FC| < 0.3 and signal >= the --ctl-quantile quantile at both time points, > 2 kb from TSS.
         The paper uses --ctl-quantile 0.5 (median) for every ChIP-seq set.
Output : rre.bed, control_pool.bed (chrom start end name lfc inj tss_dist), consensus.bed, summary.json.

Run inside the Kaggle notebooks in notebooks/ (zebrafish heart, killifish fin, mouse P1/P8) and on the
zebrafish fin BAM files (see README). Requires bedtools and samtools on PATH.
"""
import argparse
import io
import json
import os
import subprocess

import numpy as np
import pandas as pd


def sh(cmd):
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--peaks", nargs="+", required=True)
    ap.add_argument("--bam0", nargs=2, required=True, help="uninjured BAM, replicate 1 and 2")
    ap.add_argument("--bam1", nargs=2, required=True, help="injured BAM, replicate 1 and 2")
    ap.add_argument("--tss", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--ctl-quantile", dest="ctl_q", type=float, default=0.75,
                    help="control signal threshold (the paper uses 0.5)")
    ap.add_argument("--chrom-regex", dest="chrom_re", default=r"^chr[0-9]+\b",
                    help="chromosomes used (UCSC: ^chr[0-9]+; NCBI: ^NC_)")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    cons = os.path.join(a.out, "consensus.bed")
    sh(f"cat {' '.join(a.peaks)} | cut -f1-3 | grep -E '{a.chrom_re}' | sort -k1,1 -k2,2n | bedtools merge -i - > {cons}")
    bams = a.bam0 + a.bam1
    cov = sh(f"bedtools multicov -bams {' '.join(bams)} -bed {cons}")
    df = pd.read_csv(io.StringIO(cov), sep="\t", header=None,
                     names=["chrom", "start", "end", "c0a", "c0b", "c1a", "c1b"])
    lib = {c: float(sh(f"samtools view -c {b}").strip()) for c, b in zip(["c0a", "c0b", "c1a", "c1b"], bams)}
    for c in lib:
        df[c] = df[c] / lib[c] * 1e6 / ((df.end - df.start) / 1000)  # CPM per kb
    ps = 0.1
    df["lfc_a"] = np.log2((df.c1a + ps) / (df.c0a + ps))
    df["lfc_b"] = np.log2((df.c1b + ps) / (df.c0b + ps))
    df["lfc"] = (df.lfc_a + df.lfc_b) / 2
    df["inj"] = (df.c1a + df.c1b) / 2
    df["uni"] = (df.c0a + df.c0b) / 2
    p75_1, p75_0 = df.inj.quantile(0.75), df.uni.quantile(0.75)
    q1, q0 = df.inj.quantile(a.ctl_q), df.uni.quantile(a.ctl_q)

    tss = pd.read_csv(a.tss, sep="\t", header=None, usecols=[0, 1], names=["chrom", "pos"])
    df["tss_dist"] = np.inf
    for c, g in df.groupby("chrom"):
        t = np.sort(tss.loc[tss.chrom == c, "pos"].values)
        if len(t) < 2:
            continue
        mid = ((g.start + g.end) // 2).values
        j = np.searchsorted(t, mid).clip(1, len(t) - 1)
        df.loc[g.index, "tss_dist"] = np.minimum(np.abs(mid - t[j - 1]), np.abs(mid - t[j]))
    distal = df.tss_dist > 2000
    rre = df[(df.lfc_a >= 1) & (df.lfc_b >= 1) & (df.inj >= p75_1) & distal].copy()
    ctl = df[(df.lfc.abs() < 0.3) & (df.inj >= q1) & (df.uni >= q0) & distal].copy()
    rre["name"] = [f"RRE_{i}" for i in range(len(rre))]
    ctl["name"] = [f"CTL_{i}" for i in range(len(ctl))]
    cols = ["chrom", "start", "end", "name", "lfc", "inj", "tss_dist"]
    rre[cols].to_csv(os.path.join(a.out, "rre.bed"), sep="\t", header=False, index=False)
    ctl[cols].to_csv(os.path.join(a.out, "control_pool.bed"), sep="\t", header=False, index=False)
    s = {"consensus_peaks": len(df), "rre": len(rre), "control_pool": len(ctl), "lib_sizes": lib,
         "p75_inj": float(p75_1), "p75_uni": float(p75_0), "ctl_quantile": a.ctl_q,
         "rep_concordance_spearman": float(df[["lfc_a", "lfc_b"]].corr(method="spearman").iloc[0, 1])}
    json.dump(s, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    print(json.dumps(s, indent=2))


if __name__ == "__main__":
    main()


## Reference: danRer10 Bowtie 2 index (iGenomes) and UCSC files

In [ ]:
%%bash
set -e
mkdir -p idx ref
IG=https://ngi-igenomes.s3.amazonaws.com/igenomes/Danio_rerio/UCSC/danRer10/Sequence/Bowtie2Index
for f in genome.1.bt2 genome.2.bt2 genome.3.bt2 genome.4.bt2 genome.rev.1.bt2 genome.rev.2.bt2; do
  [ -s idx/$f ] || wget -q $IG/$f -O idx/$f
done
U=https://hgdownload.soe.ucsc.edu/goldenPath
wget -q $U/danRer10/bigZips/danRer10.2bit -O ref/genome.2bit
wget -q $U/danRer10/bigZips/danRer10.chrom.sizes -O ref/genome.sizes
wget -q $U/danRer10/database/refGene.txt.gz -O ref/refGene.txt.gz
zcat ref/refGene.txt.gz | awk -F'\t' 'BEGIN{OFS="\t"}{p=($4=="+")?$5:$6; print $3,p,p+1,$13,".",$4}' | sort -u > ref/tss.bed
bowtie2-inspect -n idx/genome | head -3   # check UCSC-style chromosome names (chr1, ...)
ls -la idx ref

## FASTQ from ENA (all zebrafish H3K27ac runs of the project, MD5-checked)

In [ ]:
import subprocess, hashlib, concurrent.futures as cf, urllib.request, csv, io, os
os.makedirs('fq', exist_ok=True)
api = ('https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA559885&result=read_run'
       '&fields=run_accession,sample_alias,experiment_title,fastq_ftp,fastq_md5&format=tsv')
rows = [r for r in csv.DictReader(io.StringIO(urllib.request.urlopen(api).read().decode()), delimiter='\t')
        if 'zebrafish' in r['experiment_title'] and 'H3K27ac' in r['experiment_title']]
def get(r):
    f = 'fq/' + os.path.basename(r['fastq_ftp'])
    for _ in range(5):
        subprocess.run(['wget', '-q', '-c', 'https://' + r['fastq_ftp'], '-O', f])
        m = hashlib.md5()
        with open(f, 'rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b''):
                m.update(chunk)
        h = m.hexdigest()
        if h == r['fastq_md5']:
            return r['run_accession'], 'OK'
        os.remove(f)
    return r['run_accession'], 'MD5_FAILED'
with cf.ThreadPoolExecutor(3) as ex:
    for res in ex.map(get, rows):
        print(*res)
for r in rows:
    print(r['run_accession'], r['sample_alias'], r['experiment_title'])

## Read processing: fastp -> Bowtie 2 -> MAPQ >= 30 -> duplicates removed
SRR11886651/SRR11886710 = 0 dpa R1/R2; SRR11886709/SRR11886708 = 1 dpa R1/R2; SRR11886654/SRR11886653 = input 0/1 dpa.

In [ ]:
%%bash
set -e
mkdir -p bam qc
for s in SRR11886651 SRR11886710 SRR11886709 SRR11886708 SRR11886654 SRR11886653; do
  [ -s bam/$s.bam ] && continue
  fastp -w 4 -i fq/$s.fastq.gz -o fq/$s.trim.fq.gz -j qc/$s.fastp.json -h qc/$s.fastp.html 2> qc/$s.fastp.log
  bowtie2 -p 4 -x idx/genome -U fq/$s.trim.fq.gz 2> qc/$s.bowtie2.log \
    | samtools view -@2 -b -q 30 -F 4 - | samtools sort -@2 -o bam/$s.sorted.bam -
  samtools markdup -@2 -r -s bam/$s.sorted.bam bam/$s.bam 2> qc/$s.markdup.log
  samtools index bam/$s.bam
  rm -f bam/$s.sorted.bam fq/$s.trim.fq.gz
  echo "$s done: $(samtools view -c bam/$s.bam) reads"; tail -1 qc/$s.bowtie2.log
done

## MACS3 broad peaks, each sample against the input of its time point

In [ ]:
%%bash
set -e
mkdir -p peaks
call() { macs3 callpeak -t bam/$1.bam -c bam/$2.bam -f BAM -g 1.37e9 --broad -n $3 --outdir peaks 2> peaks/$3.log; }
call SRR11886651 SRR11886654 d0_R1
call SRR11886710 SRR11886654 d0_R2
call SRR11886709 SRR11886653 d1_R1
call SRR11886708 SRR11886653 d1_R2
wc -l peaks/*.broadPeak

## Enhancers (control signal >= median)

In [ ]:
%%bash
set -e
python 01_call_injury_responsive_enhancers.py --peaks peaks/d0_R1_peaks.broadPeak peaks/d0_R2_peaks.broadPeak \
  peaks/d1_R1_peaks.broadPeak peaks/d1_R2_peaks.broadPeak \
  --bam0 bam/SRR11886651.bam bam/SRR11886710.bam --bam1 bam/SRR11886709.bam bam/SRR11886708.bam \
  --tss ref/tss.bed --ctl-quantile 0.5 --out enhancers
tar czf enhancers_results.tgz enhancers qc peaks/*.broadPeak peaks/*.log
rm -rf fq idx